In [1]:
import os, subprocess
from pathlib import Path
import pandas as pd

DATA_DIR = Path(os.environ.get("DATA_DIR", "/scratch/sm3664/medicare-fraud"))
RAW = DATA_DIR / "raw" / "part_b"
OUT = DATA_DIR / "processed" / "part_b_1319"
AUDIT = OUT / "audit"
AUDIT.mkdir(parents=True, exist_ok=True)

YEARS = list(range(2013, 2020))
KINDS = ["Provider_and_Service", "Provider"]

def raw_path(kind, year):
    name = f"Medicare_Physician_Other_Practitioners_by_{kind}_{year}"
    return RAW / name / f"{name}.csv"

In [2]:
rows = []
for y in YEARS:
    for k in KINDS:
        p = raw_path(k, y)
        rows.append({"year": y, "kind": k, "exists": p.exists(),
                     "size_gb": round(p.stat().st_size / 1e9, 2) if p.exists() else None})
print(pd.DataFrame(rows).to_string(index=False))

 year                 kind  exists  size_gb
 2013 Provider_and_Service    True     2.99
 2013             Provider    True     0.35
 2014 Provider_and_Service    True     2.98
 2014             Provider    True     0.36
 2015 Provider_and_Service    True     3.06
 2015             Provider    True     0.37
 2016 Provider_and_Service    True     3.14
 2016             Provider    True     0.39
 2017 Provider_and_Service    True     3.17
 2017             Provider    True     0.48
 2018 Provider_and_Service    True     3.21
 2018             Provider    True     0.49
 2019 Provider_and_Service    True     3.26
 2019             Provider    True     0.51


In [3]:
headers = {(k, y): list(pd.read_csv(raw_path(k, y), nrows=0, encoding="latin-1").columns)
           for y in YEARS for k in KINDS}

for k in KINDS:
    base = headers[(k, 2019)]
    print(f"\n{k}: 2019 has {len(base)} columns")
    for y in YEARS:
        cols = headers[(k, y)]
        missing = [c for c in base if c not in cols]
        extra = [c for c in cols if c not in base]
        print(y, len(cols), "| same order" if cols == base else "| order differs",
              "| missing:", missing, "| extra:", extra)


Provider_and_Service: 2019 has 28 columns
2013 28 | same order | missing: [] | extra: []
2014 28 | same order | missing: [] | extra: []
2015 28 | same order | missing: [] | extra: []
2016 28 | same order | missing: [] | extra: []
2017 28 | same order | missing: [] | extra: []
2018 28 | same order | missing: [] | extra: []
2019 28 | same order | missing: [] | extra: []

Provider: 2019 has 81 columns
2013 56 | order differs | missing: ['Bene_CC_BH_ADHD_OthCD_V1_Pct', 'Bene_CC_BH_Alcohol_Drug_V1_Pct', 'Bene_CC_BH_Tobacco_V1_Pct', 'Bene_CC_BH_Alz_NonAlzdem_V2_Pct', 'Bene_CC_BH_Anxiety_V1_Pct', 'Bene_CC_BH_Bipolar_V1_Pct', 'Bene_CC_BH_Mood_V2_Pct', 'Bene_CC_BH_Depress_V1_Pct', 'Bene_CC_BH_PD_V1_Pct', 'Bene_CC_BH_PTSD_V1_Pct', 'Bene_CC_BH_Schizo_OthPsy_V1_Pct', 'Bene_CC_PH_Asthma_V2_Pct', 'Bene_CC_PH_Afib_V2_Pct', 'Bene_CC_PH_Cancer6_V2_Pct', 'Bene_CC_PH_CKD_V2_Pct', 'Bene_CC_PH_COPD_V2_Pct', 'Bene_CC_PH_Diabetes_V2_Pct', 'Bene_CC_PH_HF_NonIHD_V2_Pct', 'Bene_CC_PH_Hyperlipidemia_V2_Pct', 'B

In [4]:
print(headers[("Provider_and_Service", 2013)])

KEY = ["Rndrng_NPI", "Rndrng_Prvdr_Gndr", "Rndrng_Prvdr_Type", "Rndrng_Prvdr_Ent_Cd",
       "Place_Of_Srvc", "HCPCS_Cd", "Tot_Benes", "Tot_Srvcs", "Tot_Bene_Day_Srvcs",
       "Avg_Sbmtd_Chrg", "Avg_Mdcr_Pymt_Amt"]
present = pd.DataFrame({y: [c in headers[("Provider_and_Service", y)] for c in KEY] for y in YEARS},
                       index=KEY)
print(present)

['Rndrng_NPI', 'Rndrng_Prvdr_Last_Org_Name', 'Rndrng_Prvdr_First_Name', 'Rndrng_Prvdr_MI', 'Rndrng_Prvdr_Crdntls', 'Rndrng_Prvdr_Ent_Cd', 'Rndrng_Prvdr_St1', 'Rndrng_Prvdr_St2', 'Rndrng_Prvdr_City', 'Rndrng_Prvdr_State_Abrvtn', 'Rndrng_Prvdr_State_FIPS', 'Rndrng_Prvdr_Zip5', 'Rndrng_Prvdr_RUCA', 'Rndrng_Prvdr_RUCA_Desc', 'Rndrng_Prvdr_Cntry', 'Rndrng_Prvdr_Type', 'Rndrng_Prvdr_Mdcr_Prtcptg_Ind', 'HCPCS_Cd', 'HCPCS_Desc', 'HCPCS_Drug_Ind', 'Place_Of_Srvc', 'Tot_Benes', 'Tot_Srvcs', 'Tot_Bene_Day_Srvcs', 'Avg_Sbmtd_Chrg', 'Avg_Mdcr_Alowd_Amt', 'Avg_Mdcr_Pymt_Amt', 'Avg_Mdcr_Stdzd_Amt']
                      2013   2014   2015   2016   2017   2018   2019
Rndrng_NPI            True   True   True   True   True   True   True
Rndrng_Prvdr_Gndr    False  False  False  False  False  False  False
Rndrng_Prvdr_Type     True   True   True   True   True   True   True
Rndrng_Prvdr_Ent_Cd   True   True   True   True   True   True   True
Place_Of_Srvc         True   True   True   True   True   True   

In [5]:
counts = []
for y in YEARS:
    for k in KINDS:
        n = int(subprocess.run(["wc", "-l", str(raw_path(k, y))],
                               capture_output=True, text=True).stdout.split()[0]) - 1
        counts.append({"year": y, "kind": k, "rows": n})
counts = pd.DataFrame(counts).pivot(index="year", columns="kind", values="rows")
print(counts)
print("Prov_Svc total:", counts["Provider_and_Service"].sum())

kind  Provider  Provider_and_Service
year                                
2013    956251               9286633
2014    986657               9316209
2015   1019377               9496848
2016   1053958               9714896
2017   1088687               9847443
2018   1121462               9961865
2019   1155870              10140228
Prov_Svc total: 67764122


In [6]:
type_counts, gender_counts = {}, {}
for y in YEARS:
    cols = [c for c in ["Rndrng_NPI", "Rndrng_Prvdr_Type", "Rndrng_Prvdr_Gndr", "Rndrng_Prvdr_Ent_Cd"]
            if c in headers[("Provider_and_Service", y)]]
    df = pd.read_csv(raw_path("Provider_and_Service", y), usecols=cols,
                     dtype=str, encoding="latin-1").drop_duplicates("Rndrng_NPI")
    type_counts[y] = df["Rndrng_Prvdr_Type"].value_counts()
    if "Rndrng_Prvdr_Gndr" in df:
        gender_counts[y] = df["Rndrng_Prvdr_Gndr"].fillna("<blank>").value_counts()
    print(y, "unique NPIs:", len(df), "| provider types:", df["Rndrng_Prvdr_Type"].nunique())

types = pd.DataFrame(type_counts).fillna(0).astype(int)
print("\nTotal unique provider types 2013-2019:", len(types), "(paper: 127 before fixes, 102 after)")
print("\nGender values per year:\n", pd.DataFrame(gender_counts))

2013 unique NPIs: 909560 | provider types: 90
2014 unique NPIs: 938141 | provider types: 91
2015 unique NPIs: 968333 | provider types: 91
2016 unique NPIs: 1000924 | provider types: 91
2017 unique NPIs: 1032911 | provider types: 94
2018 unique NPIs: 1061278 | provider types: 99
2019 unique NPIs: 1093367 | provider types: 99

Total unique provider types 2013-2019: 127 (paper: 127 before fixes, 102 after)

Gender values per year:
 Empty DataFrame
Columns: []
Index: []


In [7]:
not_all_years = types[(types == 0).any(axis=1)]
print(not_all_years.to_string())
types.to_csv(AUDIT / "provider_type_by_year.csv")
counts.to_csv(AUDIT / "row_counts.csv")
print("Saved to", AUDIT)

                                                          2013   2014   2015   2016   2017   2018   2019
Rndrng_Prvdr_Type                                                                                       
Advanced Heart Failure and Transplant Cardiology             0      0      0      0     25    478    561
Allergy/ Immunology                                          0      0      0   3242   3278   3266   3256
Allergy/Immunology                                        3159   3186   3218      0      0      0      0
Ambulance Service Provider                                   0      0      0   9611   9606   9559   9554
Ambulance Service Supplier                                9746   9736   9679      0      0      0      0
Anesthesiologist Assistants                               1003   1170   1305      0      0      0      0
Anesthesiology Assistant                                     0      0      0   1433   1586   1646   1777
Audiologist                                            